In [16]:
Features = [
    'item_nbr',
    'family',
    'class',
    'perishable',
    'has_float_sales',
    'is_likely_closed',
    'day_of_week',
    'is_weekend',
    'is_month_start',
    'is_month_end',
    'item_mean_sales',
    'item_std_sales',
    'sales_vs_family_mean',
    'lag_1',
    'lag_3',
    'lag_5',
    'lag_7',
    'lag_14',
    'lag_30',
    'lag_60',
    'lag_140',
    'rolling_mean_3',
    'rolling_mean_7',
    'rolling_mean_14',
    'rolling_mean_30',
    'rolling_mean_140',
    'rolling_std_7',
    'rolling_std_14',
    'rolling_std_30',
    'rolling_max_7',
    'rolling_min_7',
    'transactions_lag_1',
    'transactions_lag_3',
    'transactions_lag_7',
    'transactions_lag_14',
    'transactions_rolling_7',
    'transactions_rolling_14',
    'transactions_rolling_30',
    'traffic_ratio',
    'oil_price_lag_1',
    'oil_price_lag_3',
    'oil_price_lag_7',
    'oil_price_rolling_mean_7',
    'oil_price_rolling_mean_30',
    'oil_price_change_7',
    'promo_lag_1',
    'promo_lag_3',
    'promo_lag_7',
    'promo_count_7',
    'promo_count_30',
    'days_since_last_promo',
    'days_since_last_holiday',
    'days_to_next_holiday',
    'Future_is_Holiday',
    'Future_is_National',
    'Future_is_Local',
    'Future_is_likely_closed',
    'Future_day_of_week',
    'Future_is_weekend',
    'Future_is_month_start',
    'Future_is_month_end',
    'Future_On_Promotion'
]
len(Features)

62

In [17]:
Target = 'target'

In [18]:
Categorical_Features = [
    'item_nbr',
    'family',
    'class'
]

In [19]:
Future_Features_Map = {
    'is_Holiday' : 'Future_is_Holiday',
    'is_National' : 'Future_is_National',
    'is_Local' : 'Future_is_Local',
    'is_likely_closed' : 'Future_is_likely_closed',
    'day_of_week' : 'Future_day_of_week',
    'is_weekend' : 'Future_is_weekend',
    'is_month_start' : 'Future_is_month_start',
    'is_month_end' : 'Future_is_month_end',
    'onpromotion' : 'Future_On_Promotion',
    'unit_sales' : 'target'
}
len(Future_Features_Map)

10

In [20]:
horizon = 28

In [21]:
from sklearn.preprocessing import FunctionTransformer
import numpy as np

transformer = FunctionTransformer(func=np.log1p, inverse_func=np.expm1)

In [22]:
from xgboost import XGBRegressor
import joblib
import polars as pl
from tqdm import trange

class RetailForecastingModel:
    def __init__(self, params, horizon):
        if params != None and len(params) != horizon:
            raise ValueError("the length of params should be the same as the horizon")
        
        self.horizon = horizon
        self.models = []
        for h in range(horizon):
            parameters = (params[h] if params != None else {}) | {
                'objective': 'reg:squarederror',
                'tree_method': 'hist',
                'device': 'cuda',
                'enable_categorical': True,
                'n_jobs': -1,
                'verbosity': 0
            }
            self.models.append(XGBRegressor(**parameters))
    
    def fit(self, Dataset : pl.DataFrame, save_models = False, save_path = None):
        for h in trange(self.horizon, desc='Currently on Model'):
            Added_Future_Features_Dataset = Dataset.with_columns(
                [
                    pl.col(oldName).shift(-(h+1)).over('item_nbr').alias(newName) for oldName, newName in Future_Features_Map.items()
                ]
            ).with_columns(
                [
                    pl.col(colName).cast(pl.String).cast(pl.Categorical) for colName in Categorical_Features
                ]
            ).with_columns(
                pl.col('item_nbr').shift(-self.horizon).over('item_nbr').alias('delete')
            )

            Added_Future_Features_Dataset = Added_Future_Features_Dataset.drop_nulls()

            X = Added_Future_Features_Dataset.select(Features).to_pandas()
            y = Added_Future_Features_Dataset.get_column(Target).to_numpy()

            self.models[h].fit(X, y)

            if save_models:
                joblib.dump(self.models[h], save_path+f'/Model{h+1}.pkl')
    
    def predict_sales(self, X : pl.DataFrame, transformer : FunctionTransformer):
        sales = []

        future_features_map = Future_Features_Map.copy()
        future_features_map.pop('unit_sales', None)

        for h in range(self.horizon):
            Added_Future_Features_X = X.with_columns(
                [
                    pl.col(oldName).shift(-(h+1)).over('item_nbr').alias(newName) for oldName, newName in future_features_map.items()
                ]
            ).with_columns(
                [
                    pl.col(colName).cast(pl.String).cast(pl.Categorical) for colName in Categorical_Features
                ]
            ).with_columns(
                pl.col('item_nbr').shift(-self.horizon).over('item_nbr').alias('delete')
            )

            Added_Future_Features_X = Added_Future_Features_X.drop_nulls()

            rounding_mask = Added_Future_Features_X.get_column('has_float_sales').to_numpy()

            Added_Future_Features_X = Added_Future_Features_X.select(Features).to_pandas()

            prediction = self.models[h].predict(Added_Future_Features_X)

            prediction = transformer.inverse_transform(prediction)

            prediction = np.clip(prediction, a_min=0, a_max=None)

            prediction = np.where(rounding_mask == 0, np.round(prediction), prediction)

            sales.append(prediction)

        return np.array(sales).T
    
    def load_models(self, models_path : str):
        for h in range(self.horizon):
            self.models[h] = joblib.load(models_path+f'/Model{h+1}.pkl')

In [23]:
def NWRMSLE_Evaluator(Dataset : pl.DataFrame, horizon : int, horizon_total : int, transformer : FunctionTransformer, y_pred_inverse_transformed = True):
    mapping = {
        0 : 1.0,
        1 : 1.5
    }
    Added_Future_Features_Dataset = Dataset.with_columns(
            [
                pl.col(oldName).shift(-(horizon)).over('item_nbr').alias(newName) for oldName, newName in Future_Features_Map.items()
            ]
        ).with_columns(
            [
                pl.col(colName).cast(pl.String).cast(pl.Categorical) for colName in Categorical_Features
            ]
        ).with_columns(
            [
                pl.col('perishable').replace_strict(mapping, default=None).alias('weight'),
                pl.col('item_nbr').shift(-horizon_total).over('item_nbr').alias('delete')
            ]
        )

    Added_Future_Features_Dataset = Added_Future_Features_Dataset.drop_nulls()

    rounding_mask = Added_Future_Features_Dataset.get_column('has_float_sales').to_numpy()

    X = Added_Future_Features_Dataset.select(Features).to_pandas()
    y = Added_Future_Features_Dataset.get_column(Target).to_numpy()
    weights = Added_Future_Features_Dataset.get_column('weight').to_numpy()
    
    def NWRMSLE(y_true, y_pred):
        if not y_pred_inverse_transformed:
            y_pred = transformer.inverse_transform(y_pred)
            y_pred = np.clip(y_pred, a_min=0, a_max=None)
            y_pred = np.where(rounding_mask == 0, np.round(y_pred), y_pred)
        y_pred_transformed = transformer.transform(y_pred)

        loss = (weights*(y_pred_transformed - y_true)**2).sum()/weights.sum()

        return loss, 
    
    return NWRMSLE, X, y

In [24]:
Train_set = pl.read_csv('./Store39_Dataset/MasterFullTrain.csv')
Test_set = pl.read_csv('./Store39_Dataset/MasterTest.csv')

In [25]:
params = []
for h in range(1, horizon+1):
    study = joblib.load(f'./optuna_studies_xgb/Optuna_Model{h}_Study.pkl')
    params.append(study.best_trial.params)

In [26]:
params = None

In [27]:
model = RetailForecastingModel(params=params, horizon=horizon)

In [28]:
# model.load_models('./Models')

In [29]:
import os
models_path = f"./Models_XGB"
os.makedirs(models_path, exist_ok=True)

In [30]:
model.fit(Dataset=Train_set, save_models=True, save_path=models_path)

Currently on Model: 100%|██████████| 28/28 [01:32<00:00,  3.29s/it]


In [31]:
predictions = model.predict_sales(Test_set, transformer)
predictions.shape

(6160, 28)

In [32]:
errors = []
for h in range(horizon):
    evaluator, X_test, y_test = NWRMSLE_Evaluator(Test_set, h+1, horizon, transformer)
    errors.append(evaluator(y_test, predictions[:,h]))
errors = np.array(errors)
errors

array([[0.40097411],
       [0.49150192],
       [0.4545956 ],
       [0.46430633],
       [0.49297878],
       [0.4896645 ],
       [0.57593033],
       [0.54063545],
       [0.49914692],
       [0.49514606],
       [0.53429961],
       [0.50522563],
       [0.5626702 ],
       [0.52382562],
       [0.50792615],
       [0.53280417],
       [0.50837002],
       [0.51598554],
       [0.55868412],
       [0.56855909],
       [0.57452984],
       [0.54449467],
       [0.61893725],
       [0.54992566],
       [0.61467487],
       [0.66034315],
       [0.56612296],
       [0.53287766]])

In [33]:
errors.mean()

np.float64(0.5316120079918262)